In [1]:
%load_ext autoreload
%autoreload 2

### 1. 모듈 임포트

In [3]:
import torch
import numpy as np
# 1. 모듈 임포트
from modules.config import config
from modules.sdf_generator import SDFGenerator
from modules.particle_sampler import sample_particles_poisson
from modules.sdf_network import FeatureConstruction
from modules.visualizer import visualize_simulation, visualize_particles_and_features, visualize_feature_grid

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
[Taichi] version 1.7.4, llvm 15.0.1, commit b4b956fd, win, python 3.9.25


In [4]:
# ==========================================
# Setup: Config & 디바이스 초기화
# ==========================================
print(f"⚙️ Config 설정: Domain Size={config.domain_size}, Resolution={config.resolution}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"⚡ 디바이스 초기화: {device}")

⚙️ Config 설정: Domain Size=2.0, Resolution=128
⚡ 디바이스 초기화: cuda


## 2. 데이터셋 생성

### 2.1 데이터 셋 sdf 생성 및 Poisson Disk 샘플링

In [ ]:
import os
import glob
from modules.dataset import SDFDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# sdf, 파티클 생성 함수.
SDFDataset.generate_dataset(
    output_dir="dataset4",
    dataset_size=40,
    start_index=0,
    save_sdf=True,
    save_particles=True,
    save_mc=True,        # mc는 데이터로더에서도 계산 가능. False해도 됨.
    config=config,
    device=device,
    cleanup_old_files = True
)

In [ ]:
import numpy as np
import torch
from modules.config import config
from modules.visualizer import visualize_simulation, visualize_feature_grid,visualize_particles_and_features

# 1. 확인할 데이터 인덱스 설정 (0 ~ 4)
sample_idx =3
sdf_file = f"dataset4/sdf_grid_{sample_idx:03d}.npy"
mc_file = f"dataset4/mc_grid_{sample_idx:03d}.npy"

# 2. 데이터 로드 완료
sdf_grid = np.load(sdf_file)
mc_grid = np.load(mc_file)
print(f"📥 SDF 데이터 로드: {sdf_file} (Shape: {sdf_grid.shape})")
print(f"📥 m_c 데이터 로드: {mc_file} (Shape: {mc_grid.shape})")

# 3. [시각화 1] Target SDF (원본 3D 지오메트리 형태)
print("\n[1] 정답(Target) SDF 형상 시각화")
visualize_simulation(
    sdf_grid=sdf_grid, 
    domain_size=config.domain_size, 
    title=f"Target SDF Shape (Sample {sample_idx})"
)

# 4. [시각화 2] Input 3D CNN Features (파티클로부터 밀도 단위로 추출된 m_c 값)
print("\n[2] 입력(Input) m_c 특징 공간 시각화")

fig2 = visualize_feature_grid(
    m_c_grid=mc_grid,         # 앞서 파이프라인에서 나온 mc_grid 텐서 그대로 삽입
    grid_nodes=grid_nodes,    # 앞서 파이프라인에서 나온 grid_nodes 텐서 그대로 삽입
    title=f"Input m_c Feature Grid (Sample {sample_idx})"
)

In [ ]:
# 1. 기존 Config 설정값 기준
domain_size = config.domain_size
resolution = config.resolution

min_bound = -domain_size / 2.0  # -1.0
max_bound = domain_size / 2.0   # 1.0

# 2. X, Y, Z 축 각각에 대해 -1.0 부터 1.0 까지 64등분한 좌표점 생성
axis_coords = torch.linspace(min_bound, max_bound, resolution)

# 3. 3차원 공간(Meshgrid)으로 확장하여 (64, 64, 64, 3) 형태의 텐서 생성
X, Y, Z = torch.meshgrid(axis_coords, axis_coords, axis_coords, indexing='ij')

# 4. 좌표들을 하나로 묶고 (262144, 3) 형태로 평탄화(Flatten)
grid_nodes = torch.stack([X, Y, Z], dim=-1).reshape(-1, 3)

print(f"✅ 복구된 grid_nodes 형태: {grid_nodes.shape}")

In [87]:
# ==========================================
# 1. 모듈 임포트 및 Taichi 엔진 초기화
# ==========================================
import taichi as ti
from modules.config import config
from modules.taichi_fluid_solver  import TaichiFluidSolver

# 🚨 시각화나 솔버를 초기화하기 전에 반드시 GPU(혹은 CPU) 런타임을 먼저 켭니다.
ti.init(arch=ti.gpu)


# (visualizer.py가 같은 경로에 있다면 바로 임포트, modules 폴더 안이라면 modules.visualizer로 임포트)
from modules.visualizer import run_realtime_visualizer 

# ==========================================
# 2. 유체 시뮬레이터 세팅
# ==========================================
res = config.resolution            # 실시간 테스트용 해상도
domain_size = config.domain_size

# 솔버 객체 생성 및 초기 물기둥(Dam Break) 배치
solver = TaichiFluidSolver(res=128, domain_size=2.0, p_per_cell=1.0)
solver.setup_initial_fluid_block()

print(f"✅ 준비된 활성 파티클 수: {solver.get_active_particle_count():,}개")

# ==========================================
# 3. 실시간 3D 뷰어 구동!
# ==========================================
# 세팅이 끝난 솔버를 통째로 뷰어에 넘겨줍니다.
solver.run_taichi_viewer()

[Taichi] Starting on arch=cuda
✅ 준비된 활성 파티클 수: 68,445개

[초고속 튜닝 SPH 뷰어 구동]


In [ ]:
import taichi as ti

ti.init(arch=ti.gpu)

# ============================================================
# 3D WCSPH Demo using Taichi
# Educational version: O(N^2) neighbor search
# ============================================================

dim = 3

# Particle block size
NX = 9
NY = 14
NZ = 9
N = NX * NY * NZ

# Simulation parameters
dt = 2.0e-4
substeps = 8

h = 0.08                  # smoothing radius
particle_spacing = 0.045
mass = 0.08

rho0 = 1000.0             # rest density
stiffness = 600.0         # pressure stiffness
viscosity = 0.08
damping = -0.45

gravity = ti.Vector([0.0, -9.8, 0.0])

# Simulation box
box_min = ti.Vector([0.05, 0.05, 0.05])
box_max = ti.Vector([0.95, 1.25, 0.95])

# Fields
x = ti.Vector.field(dim, dtype=ti.f32, shape=N)
v = ti.Vector.field(dim, dtype=ti.f32, shape=N)
a = ti.Vector.field(dim, dtype=ti.f32, shape=N)

rho = ti.field(dtype=ti.f32, shape=N)
pressure = ti.field(dtype=ti.f32, shape=N)

# For rendering
particle_color = ti.Vector.field(3, dtype=ti.f32, shape=N)


@ti.func
def poly6_kernel(r: ti.f32) -> ti.f32:
    """
    3D Poly6 kernel:
    W(r, h) = 315 / (64*pi*h^9) * (h^2 - r^2)^3
    """
    result = 0.0
    if 0.0 <= r <= h:
        h2 = h * h
        diff = h2 - r * r
        result = 315.0 / (64.0 * ti.math.pi * h**9) * diff**3
    return result


@ti.func
def spiky_grad(r_vec: ti.template()) -> ti.types.vector(3, ti.f32):
    """
    Gradient of 3D Spiky kernel:
    grad W = -45 / (pi*h^6) * (h-r)^2 * r_vec / r
    """
    result = ti.Vector([0.0, 0.0, 0.0])
    r = r_vec.norm()

    if 1.0e-6 < r <= h:
        result = -45.0 / (ti.math.pi * h**6) * (h - r)**2 * r_vec / r

    return result


@ti.func
def viscosity_laplacian(r: ti.f32) -> ti.f32:
    """
    Laplacian of 3D viscosity kernel:
    lap W = 45 / (pi*h^6) * (h-r)
    """
    result = 0.0
    if 0.0 <= r <= h:
        result = 45.0 / (ti.math.pi * h**6) * (h - r)
    return result


@ti.kernel
def init_particles():
    for i in range(N):
        ix = i % NX
        iy = (i // NX) % NY
        iz = i // (NX * NY)

        # Initial fluid block
        x[i] = ti.Vector([
            0.25 + ix * particle_spacing,
            0.15 + iy * particle_spacing,
            0.25 + iz * particle_spacing,
        ])

        v[i] = ti.Vector([0.0, 0.0, 0.0])
        a[i] = ti.Vector([0.0, 0.0, 0.0])

        # Blue-ish particles
        particle_color[i] = ti.Vector([0.1, 0.45, 1.0])


@ti.kernel
def compute_density_pressure():
    for i in range(N):
        density = 0.0

        for j in range(N):
            r = (x[i] - x[j]).norm()
            density += mass * poly6_kernel(r)

        rho[i] = density

        # Equation of state
        # Negative pressure is clamped to reduce tensile instability
        pressure[i] = stiffness * ti.max(rho[i] - rho0, 0.0)


@ti.kernel
def compute_forces():
    for i in range(N):
        pressure_force = ti.Vector([0.0, 0.0, 0.0])
        viscosity_force = ti.Vector([0.0, 0.0, 0.0])

        for j in range(N):
            if i != j:
                rij = x[i] - x[j]
                r = rij.norm()

                if r < h:
                    # Pressure force
                    grad_w = spiky_grad(rij)

                    pressure_force += -mass * (
                        pressure[i] / (rho[i] * rho[i] + 1.0e-6)
                        + pressure[j] / (rho[j] * rho[j] + 1.0e-6)
                    ) * grad_w

                    # Viscosity force
                    lap_w = viscosity_laplacian(r)
                    viscosity_force += viscosity * mass * (
                        v[j] - v[i]
                    ) / (rho[j] + 1.0e-6) * lap_w

        a[i] = pressure_force + viscosity_force + gravity


@ti.kernel
def integrate():
    for i in range(N):
        v[i] += dt * a[i]
        x[i] += dt * v[i]

        # Boundary collision
        for d in ti.static(range(3)):
            if x[i][d] < box_min[d]:
                x[i][d] = box_min[d]
                v[i][d] *= damping

            if x[i][d] > box_max[d]:
                x[i][d] = box_max[d]
                v[i][d] *= damping


@ti.kernel
def update_color_by_height():
    for i in range(N):
        t = (x[i].y - box_min.y) / (box_max.y - box_min.y)
        t = ti.max(0.0, ti.min(1.0, t))

        # 아래쪽은 진한 파랑, 위쪽은 밝은 하늘색
        particle_color[i] = ti.Vector([
            0.05 + 0.25 * t,
            0.25 + 0.50 * t,
            1.00,
        ])


def main():
    init_particles()

    window = ti.ui.Window("3D SPH Fluid Simulation - Taichi", (1280, 720))
    canvas = window.get_canvas()
    scene = window.get_scene()

    camera = ti.ui.Camera()
    camera.position(1.4, 1.0, 1.8)
    camera.lookat(0.5, 0.45, 0.5)
    camera.up(0.0, 1.0, 0.0)
    camera.fov(45)

    paused = False

    while window.running:
        if window.get_event(ti.ui.PRESS):
            if window.event.key == ti.ui.SPACE:
                paused = not paused
            elif window.event.key == "r":
                init_particles()

        if not paused:
            for _ in range(substeps):
                compute_density_pressure()
                compute_forces()
                integrate()

            update_color_by_height()

        camera.track_user_inputs(
            window,
            movement_speed=0.02,
            hold_key=ti.ui.RMB,
        )

        scene.set_camera(camera)
        scene.ambient_light((0.65, 0.65, 0.65))
        scene.point_light(pos=(1.5, 2.0, 1.5), color=(1.0, 1.0, 1.0))

        canvas.set_background_color((1.0, 1.0, 1.0))

        # Render particles
        scene.particles(
            x,
            radius=0.012,
            per_vertex_color=particle_color,
        )

        canvas.scene(scene)

        gui = window.get_gui()
        gui.text("3D WCSPH Demo")
        gui.text("Right mouse drag: rotate camera")
        gui.text("W/A/S/D/Q/E: move camera")
        gui.text("SPACE: pause / resume")
        gui.text("R: reset particles")
        gui.text(f"Particles: {N}")

        window.show()


if __name__ == "__main__":
    main()

[Taichi] Starting on arch=cuda


In [72]:
import taichi as ti
import numpy as np
from skimage import measure

ti.init(arch=ti.gpu)

# ============================================================
# 3D SPH + Marching Cubes Surface Visualization
# Demo version
# ============================================================

dim = 3

# ------------------------------------------------------------
# Particle settings
# ------------------------------------------------------------
NX = 8
NY = 12
NZ = 8
N = NX * NY * NZ

dt = 2.0e-4
substeps = 8

h = 0.08
particle_spacing = 0.045
mass = 0.08

rho0 = 1000.0
stiffness = 600.0
viscosity = 0.08
damping = -0.45

gravity = ti.Vector([0.0, -9.8, 0.0])

box_min_np = np.array([0.05, 0.05, 0.05], dtype=np.float32)
box_max_np = np.array([0.95, 1.25, 0.95], dtype=np.float32)

box_min = ti.Vector([0.05, 0.05, 0.05])
box_max = ti.Vector([0.95, 1.25, 0.95])

# ------------------------------------------------------------
# Marching Cubes settings
# ------------------------------------------------------------
GRID_RES = 42
SURFACE_RADIUS = h * 1.8
ISO_LEVEL = 0.45
MESH_UPDATE_INTERVAL = 4

# ------------------------------------------------------------
# Taichi fields
# ------------------------------------------------------------
x = ti.Vector.field(dim, dtype=ti.f32, shape=N)
v = ti.Vector.field(dim, dtype=ti.f32, shape=N)
a = ti.Vector.field(dim, dtype=ti.f32, shape=N)

rho = ti.field(dtype=ti.f32, shape=N)
pressure = ti.field(dtype=ti.f32, shape=N)

particle_color = ti.Vector.field(3, dtype=ti.f32, shape=N)


@ti.func
def poly6_kernel(r: ti.f32) -> ti.f32:
    result = 0.0
    if 0.0 <= r <= h:
        h2 = h * h
        diff = h2 - r * r
        result = 315.0 / (64.0 * ti.math.pi * h**9) * diff**3
    return result


@ti.func
def spiky_grad(r_vec: ti.template()) -> ti.types.vector(3, ti.f32):
    result = ti.Vector([0.0, 0.0, 0.0])
    r = r_vec.norm()

    if 1.0e-6 < r <= h:
        result = -45.0 / (ti.math.pi * h**6) * (h - r)**2 * r_vec / r

    return result


@ti.func
def viscosity_laplacian(r: ti.f32) -> ti.f32:
    result = 0.0
    if 0.0 <= r <= h:
        result = 45.0 / (ti.math.pi * h**6) * (h - r)
    return result


@ti.kernel
def init_particles():
    for i in range(N):
        ix = i % NX
        iy = (i // NX) % NY
        iz = i // (NX * NY)

        x[i] = ti.Vector([
            0.28 + ix * particle_spacing,
            0.15 + iy * particle_spacing,
            0.28 + iz * particle_spacing,
        ])

        v[i] = ti.Vector([0.0, 0.0, 0.0])
        a[i] = ti.Vector([0.0, 0.0, 0.0])

        particle_color[i] = ti.Vector([0.05, 0.35, 1.0])


@ti.kernel
def compute_density_pressure():
    for i in range(N):
        density = 0.0

        for j in range(N):
            r = (x[i] - x[j]).norm()
            density += mass * poly6_kernel(r)

        rho[i] = density

        # WCSPH equation of state
        pressure[i] = stiffness * ti.max(rho[i] - rho0, 0.0)


@ti.kernel
def compute_forces():
    for i in range(N):
        pressure_force = ti.Vector([0.0, 0.0, 0.0])
        viscosity_force = ti.Vector([0.0, 0.0, 0.0])

        for j in range(N):
            if i != j:
                rij = x[i] - x[j]
                r = rij.norm()

                if r < h:
                    grad_w = spiky_grad(rij)

                    pressure_force += -mass * (
                        pressure[i] / (rho[i] * rho[i] + 1.0e-6)
                        + pressure[j] / (rho[j] * rho[j] + 1.0e-6)
                    ) * grad_w

                    lap_w = viscosity_laplacian(r)
                    viscosity_force += viscosity * mass * (
                        v[j] - v[i]
                    ) / (rho[j] + 1.0e-6) * lap_w

        a[i] = pressure_force + viscosity_force + gravity


@ti.kernel
def integrate():
    for i in range(N):
        v[i] += dt * a[i]
        x[i] += dt * v[i]

        # Box collision
        for d in ti.static(range(3)):
            if x[i][d] < box_min[d]:
                x[i][d] = box_min[d]
                v[i][d] *= damping

            if x[i][d] > box_max[d]:
                x[i][d] = box_max[d]
                v[i][d] *= damping


@ti.kernel
def update_particle_color_by_height():
    for i in range(N):
        t = (x[i].y - box_min.y) / (box_max.y - box_min.y)
        t = ti.max(0.0, ti.min(1.0, t))

        particle_color[i] = ti.Vector([
            0.05 + 0.20 * t,
            0.30 + 0.45 * t,
            1.00,
        ])


def build_scalar_field(particles_np):
    """
    입자 위치로부터 scalar density field 생성.
    여기서는 SPH density field를 정확히 쓰기보다,
    시각화를 위한 metaball-style field를 사용한다.

    scalar field가 ISO_LEVEL 이상인 부분이 물 표면 내부로 간주된다.
    """

    grid = np.zeros((GRID_RES, GRID_RES, GRID_RES), dtype=np.float32)

    grid_min = box_min_np
    grid_max = box_max_np
    grid_size = grid_max - grid_min

    dx = grid_size / (GRID_RES - 1)

    radius = SURFACE_RADIUS
    radius2 = radius * radius

    for p in particles_np:
        rel_min = (p - radius - grid_min) / dx
        rel_max = (p + radius - grid_min) / dx

        ix0 = max(int(np.floor(rel_min[0])), 0)
        iy0 = max(int(np.floor(rel_min[1])), 0)
        iz0 = max(int(np.floor(rel_min[2])), 0)

        ix1 = min(int(np.ceil(rel_max[0])) + 1, GRID_RES)
        iy1 = min(int(np.ceil(rel_max[1])) + 1, GRID_RES)
        iz1 = min(int(np.ceil(rel_max[2])) + 1, GRID_RES)

        if ix0 >= ix1 or iy0 >= iy1 or iz0 >= iz1:
            continue

        xs = grid_min[0] + np.arange(ix0, ix1, dtype=np.float32) * dx[0]
        ys = grid_min[1] + np.arange(iy0, iy1, dtype=np.float32) * dx[1]
        zs = grid_min[2] + np.arange(iz0, iz1, dtype=np.float32) * dx[2]

        X, Y, Z = np.meshgrid(xs, ys, zs, indexing="ij")

        dist2 = (X - p[0]) ** 2 + (Y - p[1]) ** 2 + (Z - p[2]) ** 2
        q = dist2 / radius2

        contribution = np.where(q < 1.0, (1.0 - q) ** 3, 0.0)

        grid[ix0:ix1, iy0:iy1, iz0:iz1] += contribution.astype(np.float32)

    return grid


def reconstruct_surface(particles_np):
    """
    scalar field에서 marching cubes를 실행하고,
    Taichi mesh rendering용 vertex field와 index field를 생성한다.
    """

    scalar_field = build_scalar_field(particles_np)

    if scalar_field.max() < ISO_LEVEL:
        return None, None, 0, 0

    grid_size = box_max_np - box_min_np
    spacing = grid_size / (GRID_RES - 1)

    verts, faces, normals, values = measure.marching_cubes(
        scalar_field,
        level=ISO_LEVEL,
        spacing=(spacing[0], spacing[1], spacing[2]),
    )

    verts = verts.astype(np.float32)
    verts += box_min_np

    faces = faces.astype(np.int32)
    indices = faces.reshape(-1)

    if len(verts) == 0 or len(indices) == 0:
        return None, None, 0, 0

    vertices_ti = ti.Vector.field(3, dtype=ti.f32, shape=len(verts))
    indices_ti = ti.field(dtype=ti.i32, shape=len(indices))

    vertices_ti.from_numpy(verts)
    indices_ti.from_numpy(indices)

    return vertices_ti, indices_ti, len(verts), len(faces)


def main():
    init_particles()

    window = ti.ui.Window(
        "3D SPH + Marching Cubes Surface - Taichi",
        (1280, 720),
    )

    canvas = window.get_canvas()
    scene = window.get_scene()

    camera = ti.ui.Camera()
    camera.position(1.45, 0.95, 1.75)
    camera.lookat(0.5, 0.45, 0.5)
    camera.up(0.0, 1.0, 0.0)
    camera.fov(45)

    paused = False
    show_particles = True
    show_surface = True

    frame = 0

    surface_vertices = None
    surface_indices = None
    surface_vertex_count = 0
    surface_face_count = 0

    while window.running:
        if window.get_event(ti.ui.PRESS):
            if window.event.key == ti.ui.SPACE:
                paused = not paused
            elif window.event.key == "r":
                init_particles()
                surface_vertices = None
                surface_indices = None
            elif window.event.key == "p":
                show_particles = not show_particles
            elif window.event.key == "m":
                show_surface = not show_surface

        if not paused:
            for _ in range(substeps):
                compute_density_pressure()
                compute_forces()
                integrate()

            update_particle_color_by_height()

            if frame % MESH_UPDATE_INTERVAL == 0:
                particles_np = x.to_numpy()
                result = reconstruct_surface(particles_np)

                surface_vertices = result[0]
                surface_indices = result[1]
                surface_vertex_count = result[2]
                surface_face_count = result[3]

            frame += 1

        camera.track_user_inputs(
            window,
            movement_speed=0.02,
            hold_key=ti.ui.RMB,
        )

        scene.set_camera(camera)
        scene.ambient_light((0.65, 0.65, 0.65))
        scene.point_light(pos=(1.4, 2.0, 1.2), color=(1.0, 1.0, 1.0))
        scene.point_light(pos=(-0.5, 1.5, -0.5), color=(0.6, 0.6, 0.6))

        canvas.set_background_color((1.0, 1.0, 1.0))

        # Surface mesh rendering
        if show_surface and surface_vertices is not None and surface_indices is not None:
            scene.mesh(
                surface_vertices,
                indices=surface_indices,
                color=(0.2, 0.65, 1.0),
                two_sided=True,
            )

        # Particle rendering
        if show_particles:
            scene.particles(
                x,
                radius=0.008,
                per_vertex_color=particle_color,
            )

        canvas.scene(scene)

        gui = window.get_gui()
        gui.text("3D SPH + Marching Cubes")
        gui.text("Right mouse drag: rotate camera")
        gui.text("W/A/S/D/Q/E: move camera")
        gui.text("SPACE: pause / resume")
        gui.text("R: reset")
        gui.text("P: show/hide particles")
        gui.text("M: show/hide marching cubes surface")
        gui.text(f"Particles: {N}")
        gui.text(f"Grid resolution: {GRID_RES}^3")
        gui.text(f"Surface vertices: {surface_vertex_count}")
        gui.text(f"Surface faces: {surface_face_count}")

        window.show()


if __name__ == "__main__":
    main()

[Taichi] Starting on arch=cuda


In [ ]:
import os
import taichi as ti
import numpy as np

ti.init(arch=ti.gpu)

# ============================================================
# 3D SPH + Regular Grid Level Set Saving Only
# No visualization
# ============================================================

dim = 3

# ------------------------------------------------------------
# Particle settings
# ------------------------------------------------------------
NX = 8
NY = 12
NZ = 8
N = NX * NY * NZ

dt = 2.0e-4
substeps = 8

h = 0.08
particle_spacing = 0.045
mass = 0.08

rho0 = 1000.0
stiffness = 600.0
viscosity = 0.08
damping = -0.45

gravity = ti.Vector([0.0, -9.8, 0.0])

box_min_np = np.array([0.05, 0.05, 0.05], dtype=np.float32)
box_max_np = np.array([0.95, 1.25, 0.95], dtype=np.float32)

box_min = ti.Vector([0.05, 0.05, 0.05])
box_max = ti.Vector([0.95, 1.25, 0.95])

# ------------------------------------------------------------
# Level set grid settings
# ------------------------------------------------------------
GRID_RES = 42

# particle metaball radius for level set generation
SURFACE_RADIUS = h * 1.8

# scalar field threshold
ISO_LEVEL = 0.45

# ------------------------------------------------------------
# Save settings
# ------------------------------------------------------------
SAVE_DIR = "levelset_output2"
os.makedirs(SAVE_DIR, exist_ok=True)

MAX_FRAMES = 300          # 총 저장할 프레임 수
SAVE_INTERVAL = 1         # 1이면 매 프레임 저장, 4면 4프레임마다 저장
SAVE_SCALAR_FIELD = True  # True면 scalar_field도 같이 저장

# ------------------------------------------------------------
# Taichi fields
# ------------------------------------------------------------
x = ti.Vector.field(dim, dtype=ti.f32, shape=N)
v = ti.Vector.field(dim, dtype=ti.f32, shape=N)
a = ti.Vector.field(dim, dtype=ti.f32, shape=N)

rho = ti.field(dtype=ti.f32, shape=N)
pressure = ti.field(dtype=ti.f32, shape=N)


@ti.func
def poly6_kernel(r: ti.f32) -> ti.f32:
    result = 0.0
    if 0.0 <= r <= h:
        h2 = h * h
        diff = h2 - r * r
        result = 315.0 / (64.0 * ti.math.pi * h**9) * diff**3
    return result


@ti.func
def spiky_grad(r_vec: ti.template()) -> ti.types.vector(3, ti.f32):
    result = ti.Vector([0.0, 0.0, 0.0])
    r = r_vec.norm()

    if 1.0e-6 < r <= h:
        result = -45.0 / (ti.math.pi * h**6) * (h - r)**2 * r_vec / r

    return result


@ti.func
def viscosity_laplacian(r: ti.f32) -> ti.f32:
    result = 0.0
    if 0.0 <= r <= h:
        result = 45.0 / (ti.math.pi * h**6) * (h - r)
    return result


@ti.kernel
def init_particles():
    for i in range(N):
        ix = i % NX
        iy = (i // NX) % NY
        iz = i // (NX * NY)

        x[i] = ti.Vector([
            0.28 + ix * particle_spacing,
            0.15 + iy * particle_spacing,
            0.28 + iz * particle_spacing,
        ])

        v[i] = ti.Vector([0.0, 0.0, 0.0])
        a[i] = ti.Vector([0.0, 0.0, 0.0])


@ti.kernel
def compute_density_pressure():
    for i in range(N):
        density = 0.0

        for j in range(N):
            r = (x[i] - x[j]).norm()
            density += mass * poly6_kernel(r)

        rho[i] = density

        # WCSPH equation of state
        pressure[i] = stiffness * ti.max(rho[i] - rho0, 0.0)


@ti.kernel
def compute_forces():
    for i in range(N):
        pressure_force = ti.Vector([0.0, 0.0, 0.0])
        viscosity_force = ti.Vector([0.0, 0.0, 0.0])

        for j in range(N):
            if i != j:
                rij = x[i] - x[j]
                r = rij.norm()

                if r < h:
                    grad_w = spiky_grad(rij)

                    pressure_force += -mass * (
                        pressure[i] / (rho[i] * rho[i] + 1.0e-6)
                        + pressure[j] / (rho[j] * rho[j] + 1.0e-6)
                    ) * grad_w

                    lap_w = viscosity_laplacian(r)
                    viscosity_force += viscosity * mass * (
                        v[j] - v[i]
                    ) / (rho[j] + 1.0e-6) * lap_w

        a[i] = pressure_force + viscosity_force + gravity


@ti.kernel
def integrate():
    for i in range(N):
        v[i] += dt * a[i]
        x[i] += dt * v[i]

        # Box collision
        for d in ti.static(range(3)):
            if x[i][d] < box_min[d]:
                x[i][d] = box_min[d]
                v[i][d] *= damping

            if x[i][d] > box_max[d]:
                x[i][d] = box_max[d]
                v[i][d] *= damping


def build_scalar_field(particles_np):
    """
    SPH particle positions -> regular grid scalar field.

    여기서는 물리적 SPH density field를 그대로 저장하는 것이 아니라,
    입자 위치 기반 metaball-style scalar field를 만든다.

    scalar_field > ISO_LEVEL 인 영역을 유체 내부로 본다.
    """

    grid = np.zeros((GRID_RES, GRID_RES, GRID_RES), dtype=np.float32)

    grid_min = box_min_np
    grid_max = box_max_np
    grid_size = grid_max - grid_min

    dx = grid_size / (GRID_RES - 1)

    radius = SURFACE_RADIUS
    radius2 = radius * radius

    for p in particles_np:
        rel_min = (p - radius - grid_min) / dx
        rel_max = (p + radius - grid_min) / dx

        ix0 = max(int(np.floor(rel_min[0])), 0)
        iy0 = max(int(np.floor(rel_min[1])), 0)
        iz0 = max(int(np.floor(rel_min[2])), 0)

        ix1 = min(int(np.ceil(rel_max[0])) + 1, GRID_RES)
        iy1 = min(int(np.ceil(rel_max[1])) + 1, GRID_RES)
        iz1 = min(int(np.ceil(rel_max[2])) + 1, GRID_RES)

        if ix0 >= ix1 or iy0 >= iy1 or iz0 >= iz1:
            continue

        xs = grid_min[0] + np.arange(ix0, ix1, dtype=np.float32) * dx[0]
        ys = grid_min[1] + np.arange(iy0, iy1, dtype=np.float32) * dx[1]
        zs = grid_min[2] + np.arange(iz0, iz1, dtype=np.float32) * dx[2]

        X, Y, Z = np.meshgrid(xs, ys, zs, indexing="ij")

        dist2 = (X - p[0]) ** 2 + (Y - p[1]) ** 2 + (Z - p[2]) ** 2
        q = dist2 / radius2

        contribution = np.where(q < 1.0, (1.0 - q) ** 3, 0.0)

        grid[ix0:ix1, iy0:iy1, iz0:iz1] += contribution.astype(np.float32)

    return grid


def build_levelset_from_particles(particles_np):
    """
    regular grid scalar field를 level set 형태로 변환한다.

    convention:
    levelset < 0 : fluid inside
    levelset = 0 : fluid surface
    levelset > 0 : fluid outside
    """

    scalar_field = build_scalar_field(particles_np)

    # scalar_field가 ISO_LEVEL보다 크면 내부.
    # 따라서 ISO_LEVEL - scalar_field를 사용하면 내부가 음수.
    levelset = ISO_LEVEL - scalar_field

    return levelset.astype(np.float32), scalar_field.astype(np.float32)


def save_levelset_regular_grid(frame_id, particles_np):
    """
    매 프레임 regular grid level set 데이터를 .npz로 저장한다.
    """

    levelset, scalar_field = build_levelset_from_particles(particles_np)

    grid_size = box_max_np - box_min_np
    dx = grid_size / (GRID_RES - 1)

    filename = os.path.join(SAVE_DIR, f"levelset_{frame_id:06d}.npz")

    if SAVE_SCALAR_FIELD:
        np.savez_compressed(
            filename,
            levelset=levelset,
            scalar_field=scalar_field,
            particle_positions=particles_np.astype(np.float32),
            grid_res=np.array([GRID_RES, GRID_RES, GRID_RES], dtype=np.int32),
            box_min=box_min_np.astype(np.float32),
            box_max=box_max_np.astype(np.float32),
            dx=dx.astype(np.float32),
            iso_level=np.array([ISO_LEVEL], dtype=np.float32),
            surface_radius=np.array([SURFACE_RADIUS], dtype=np.float32),
            frame=np.array([frame_id], dtype=np.int32),
            convention=np.array([
                "levelset < 0: inside, levelset = 0: surface, levelset > 0: outside"
            ]),
        )
    else:
        np.savez_compressed(
            filename,
            levelset=levelset,
            particle_positions=particles_np.astype(np.float32),
            grid_res=np.array([GRID_RES, GRID_RES, GRID_RES], dtype=np.int32),
            box_min=box_min_np.astype(np.float32),
            box_max=box_max_np.astype(np.float32),
            dx=dx.astype(np.float32),
            iso_level=np.array([ISO_LEVEL], dtype=np.float32),
            surface_radius=np.array([SURFACE_RADIUS], dtype=np.float32),
            frame=np.array([frame_id], dtype=np.int32),
            convention=np.array([
                "levelset < 0: inside, levelset = 0: surface, levelset > 0: outside"
            ]),
        )

    print(
        f"[saved] frame={frame_id:06d}, "
        f"file={filename}, "
        f"levelset min={levelset.min():.4f}, "
        f"max={levelset.max():.4f}"
    )


def simulate_one_frame():
    """
    한 프레임 동안 SPH substep 수행.
    """

    for _ in range(substeps):
        compute_density_pressure()
        compute_forces()
        integrate()


def main():
    init_particles()

    print("==============================================")
    print("3D SPH Level Set Saving Simulation")
    print("No visualization")
    print(f"Particles        : {N}")
    print(f"Grid resolution  : {GRID_RES} x {GRID_RES} x {GRID_RES}")
    print(f"Max frames       : {MAX_FRAMES}")
    print(f"Save interval    : {SAVE_INTERVAL}")
    print(f"Save directory   : {SAVE_DIR}")
    print("==============================================")

    for frame in range(MAX_FRAMES):
        simulate_one_frame()

        if frame % SAVE_INTERVAL == 0:
            particles_np = x.to_numpy()
            save_levelset_regular_grid(frame, particles_np)

    print("Simulation finished.")
    print(f"Saved level set files to: {SAVE_DIR}")


if __name__ == "__main__":
    main()

In [ ]:
import os
import taichi as ti
import numpy as np

ti.init(arch=ti.gpu)

# ============================================================
# 3D SPH + Regular Grid Level Set Saving Only
# Optimized with uniform grid neighbor search
# No visualization
# ============================================================

dim = 3

# ------------------------------------------------------------
# Particle settings
# ------------------------------------------------------------
NX = 8
NY = 12
NZ = 8
N = NX * NY * NZ

dt = 2.0e-4
substeps = 8

h = 0.08
particle_spacing = 0.045
mass = 0.08

rho0 = 1000.0
stiffness = 600.0
viscosity = 0.08
damping = -0.45

gravity = ti.Vector([0.0, -9.8, 0.0])

box_min_np = np.array([0.05, 0.05, 0.05], dtype=np.float32)
box_max_np = np.array([0.95, 1.25, 0.95], dtype=np.float32)

box_min = ti.Vector([0.05, 0.05, 0.05])
box_max = ti.Vector([0.95, 1.25, 0.95])
box_size = box_max - box_min

# ------------------------------------------------------------
# Uniform grid settings for SPH neighbor search
# ------------------------------------------------------------
CELL_SIZE = h

GRID_NX = int(np.ceil((box_max_np[0] - box_min_np[0]) / CELL_SIZE))
GRID_NY = int(np.ceil((box_max_np[1] - box_min_np[1]) / CELL_SIZE))
GRID_NZ = int(np.ceil((box_max_np[2] - box_min_np[2]) / CELL_SIZE))

MAX_PARTICLES_PER_CELL = 96

# ------------------------------------------------------------
# Level set regular grid settings
# ------------------------------------------------------------
LEVELSET_RES = 42

SURFACE_RADIUS = h * 1.8
ISO_LEVEL = 0.45

LEVELSET_SUPPORT_CELLS = int(np.ceil(SURFACE_RADIUS / CELL_SIZE))

levelset_dx_np = (box_max_np - box_min_np) / (LEVELSET_RES - 1)
levelset_dx = ti.Vector([
    float(levelset_dx_np[0]),
    float(levelset_dx_np[1]),
    float(levelset_dx_np[2]),
])

# ------------------------------------------------------------
# Save settings
# ------------------------------------------------------------
SAVE_DIR = "levelset_output"
os.makedirs(SAVE_DIR, exist_ok=True)

MAX_FRAMES = 300
SAVE_INTERVAL = 1

SAVE_PARTICLE_POSITIONS = True

# ------------------------------------------------------------
# Taichi particle fields
# ------------------------------------------------------------
x = ti.Vector.field(dim, dtype=ti.f32, shape=N)
v = ti.Vector.field(dim, dtype=ti.f32, shape=N)
a = ti.Vector.field(dim, dtype=ti.f32, shape=N)

rho = ti.field(dtype=ti.f32, shape=N)
pressure = ti.field(dtype=ti.f32, shape=N)

# ------------------------------------------------------------
# Taichi uniform grid fields
# ------------------------------------------------------------
grid_count = ti.field(
    dtype=ti.i32,
    shape=(GRID_NX, GRID_NY, GRID_NZ),
)

grid_particles = ti.field(
    dtype=ti.i32,
    shape=(GRID_NX, GRID_NY, GRID_NZ, MAX_PARTICLES_PER_CELL),
)

# ------------------------------------------------------------
# Taichi level set field
# ------------------------------------------------------------
levelset_grid = ti.field(
    dtype=ti.f32,
    shape=(LEVELSET_RES, LEVELSET_RES, LEVELSET_RES),
)


@ti.func
def poly6_kernel(r: ti.f32) -> ti.f32:
    result = 0.0

    if 0.0 <= r <= h:
        h2 = h * h
        diff = h2 - r * r
        result = 315.0 / (64.0 * ti.math.pi * h**9) * diff**3

    return result


@ti.func
def spiky_grad(r_vec: ti.template()) -> ti.types.vector(3, ti.f32):
    result = ti.Vector([0.0, 0.0, 0.0])
    r = r_vec.norm()

    if 1.0e-6 < r <= h:
        result = -45.0 / (ti.math.pi * h**6) * (h - r)**2 * r_vec / r

    return result


@ti.func
def viscosity_laplacian(r: ti.f32) -> ti.f32:
    result = 0.0

    if 0.0 <= r <= h:
        result = 45.0 / (ti.math.pi * h**6) * (h - r)

    return result


@ti.func
def get_cell(pos: ti.template()) -> ti.types.vector(3, ti.i32):
    rel = (pos - box_min) / CELL_SIZE
    cell = ti.cast(rel, ti.i32)

    cell[0] = ti.max(0, ti.min(GRID_NX - 1, cell[0]))
    cell[1] = ti.max(0, ti.min(GRID_NY - 1, cell[1]))
    cell[2] = ti.max(0, ti.min(GRID_NZ - 1, cell[2]))

    return cell


@ti.kernel
def init_particles():
    for i in range(N):
        ix = i % NX
        iy = (i // NX) % NY
        iz = i // (NX * NY)

        x[i] = ti.Vector([
            0.28 + ix * particle_spacing,
            0.15 + iy * particle_spacing,
            0.28 + iz * particle_spacing,
        ])

        v[i] = ti.Vector([0.0, 0.0, 0.0])
        a[i] = ti.Vector([0.0, 0.0, 0.0])
        rho[i] = rho0
        pressure[i] = 0.0


@ti.kernel
def clear_uniform_grid():
    for I in ti.grouped(grid_count):
        grid_count[I] = 0


@ti.kernel
def build_uniform_grid():
    for i in range(N):
        cell = get_cell(x[i])

        cx = cell[0]
        cy = cell[1]
        cz = cell[2]

        old_count = ti.atomic_add(grid_count[cx, cy, cz], 1)

        if old_count < MAX_PARTICLES_PER_CELL:
            grid_particles[cx, cy, cz, old_count] = i


@ti.kernel
def compute_density_pressure_with_grid():
    for i in range(N):
        density = 0.0

        base = get_cell(x[i])

        for ox in ti.static(range(-1, 2)):
            for oy in ti.static(range(-1, 2)):
                for oz in ti.static(range(-1, 2)):
                    cx = base[0] + ox
                    cy = base[1] + oy
                    cz = base[2] + oz

                    if 0 <= cx < GRID_NX and 0 <= cy < GRID_NY and 0 <= cz < GRID_NZ:
                        count = grid_count[cx, cy, cz]
                        count = ti.min(count, MAX_PARTICLES_PER_CELL)

                        for k in range(count):
                            j = grid_particles[cx, cy, cz, k]
                            r = (x[i] - x[j]).norm()

                            if r < h:
                                density += mass * poly6_kernel(r)

        rho[i] = density

        # WCSPH equation of state
        pressure[i] = stiffness * ti.max(rho[i] - rho0, 0.0)


@ti.kernel
def compute_forces_with_grid():
    for i in range(N):
        pressure_force = ti.Vector([0.0, 0.0, 0.0])
        viscosity_force = ti.Vector([0.0, 0.0, 0.0])

        base = get_cell(x[i])

        for ox in ti.static(range(-1, 2)):
            for oy in ti.static(range(-1, 2)):
                for oz in ti.static(range(-1, 2)):
                    cx = base[0] + ox
                    cy = base[1] + oy
                    cz = base[2] + oz

                    if 0 <= cx < GRID_NX and 0 <= cy < GRID_NY and 0 <= cz < GRID_NZ:
                        count = grid_count[cx, cy, cz]
                        count = ti.min(count, MAX_PARTICLES_PER_CELL)

                        for k in range(count):
                            j = grid_particles[cx, cy, cz, k]

                            if i != j:
                                rij = x[i] - x[j]
                                r = rij.norm()

                                if r < h:
                                    grad_w = spiky_grad(rij)

                                    pressure_force += -mass * (
                                        pressure[i] / (rho[i] * rho[i] + 1.0e-6)
                                        + pressure[j] / (rho[j] * rho[j] + 1.0e-6)
                                    ) * grad_w

                                    lap_w = viscosity_laplacian(r)

                                    viscosity_force += viscosity * mass * (
                                        v[j] - v[i]
                                    ) / (rho[j] + 1.0e-6) * lap_w

        a[i] = pressure_force + viscosity_force + gravity


@ti.kernel
def integrate():
    for i in range(N):
        v[i] += dt * a[i]
        x[i] += dt * v[i]

        for d in ti.static(range(3)):
            if x[i][d] < box_min[d]:
                x[i][d] = box_min[d]
                v[i][d] *= damping

            if x[i][d] > box_max[d]:
                x[i][d] = box_max[d]
                v[i][d] *= damping


@ti.kernel
def compute_levelset_with_grid():
    """
    현재 particle position과 uniform grid를 이용해
    regular grid level set을 병렬 계산한다.

    levelset < 0 : fluid inside
    levelset = 0 : surface
    levelset > 0 : outside
    """

    for I in ti.grouped(levelset_grid):
        grid_pos = box_min + ti.cast(I, ti.f32) * levelset_dx

        scalar = 0.0

        base = get_cell(grid_pos)

        for ox in ti.static(range(-LEVELSET_SUPPORT_CELLS, LEVELSET_SUPPORT_CELLS + 1)):
            for oy in ti.static(range(-LEVELSET_SUPPORT_CELLS, LEVELSET_SUPPORT_CELLS + 1)):
                for oz in ti.static(range(-LEVELSET_SUPPORT_CELLS, LEVELSET_SUPPORT_CELLS + 1)):
                    cx = base[0] + ox
                    cy = base[1] + oy
                    cz = base[2] + oz

                    if 0 <= cx < GRID_NX and 0 <= cy < GRID_NY and 0 <= cz < GRID_NZ:
                        count = grid_count[cx, cy, cz]
                        count = ti.min(count, MAX_PARTICLES_PER_CELL)

                        for k in range(count):
                            j = grid_particles[cx, cy, cz, k]

                            dist2 = (grid_pos - x[j]).norm_sqr()
                            radius2 = SURFACE_RADIUS * SURFACE_RADIUS

                            if dist2 < radius2:
                                q = dist2 / radius2
                                scalar += (1.0 - q) ** 3

        levelset_grid[I] = ISO_LEVEL - scalar


def rebuild_grid_for_current_positions():
    clear_uniform_grid()
    build_uniform_grid()


def simulate_one_frame():
    for _ in range(substeps):
        rebuild_grid_for_current_positions()
        compute_density_pressure_with_grid()
        compute_forces_with_grid()
        integrate()


def save_levelset_npz(frame_id):
    """
    현재 프레임의 regular grid level set을 .npz로 저장한다.
    """

    # integrate 이후 particle 위치가 바뀌었으므로,
    # level set 계산 직전에 uniform grid를 다시 만든다.
    rebuild_grid_for_current_positions()

    compute_levelset_with_grid()

    levelset_np = levelset_grid.to_numpy().astype(np.float32)

    grid_size_np = box_max_np - box_min_np
    dx_np = grid_size_np / (LEVELSET_RES - 1)

    filename = os.path.join(SAVE_DIR, f"levelset_{frame_id:06d}.npz")

    if SAVE_PARTICLE_POSITIONS:
        particles_np = x.to_numpy().astype(np.float32)

        np.savez_compressed(
            filename,
            levelset=levelset_np,
            particle_positions=particles_np,
            grid_res=np.array(
                [LEVELSET_RES, LEVELSET_RES, LEVELSET_RES],
                dtype=np.int32,
            ),
            box_min=box_min_np.astype(np.float32),
            box_max=box_max_np.astype(np.float32),
            dx=dx_np.astype(np.float32),
            iso_level=np.array([ISO_LEVEL], dtype=np.float32),
            surface_radius=np.array([SURFACE_RADIUS], dtype=np.float32),
            frame=np.array([frame_id], dtype=np.int32),
            convention=np.array([
                "levelset < 0: inside, levelset = 0: surface, levelset > 0: outside"
            ]),
        )
    else:
        np.savez_compressed(
            filename,
            levelset=levelset_np,
            grid_res=np.array(
                [LEVELSET_RES, LEVELSET_RES, LEVELSET_RES],
                dtype=np.int32,
            ),
            box_min=box_min_np.astype(np.float32),
            box_max=box_max_np.astype(np.float32),
            dx=dx_np.astype(np.float32),
            iso_level=np.array([ISO_LEVEL], dtype=np.float32),
            surface_radius=np.array([SURFACE_RADIUS], dtype=np.float32),
            frame=np.array([frame_id], dtype=np.int32),
            convention=np.array([
                "levelset < 0: inside, levelset = 0: surface, levelset > 0: outside"
            ]),
        )

    print(
        f"[saved] frame={frame_id:06d}, "
        f"file={filename}, "
        f"min={levelset_np.min():.4f}, "
        f"max={levelset_np.max():.4f}"
    )


def main():
    init_particles()

    print("==============================================")
    print("3D SPH Level Set Saving Simulation")
    print("No visualization")
    print("Uniform grid neighbor search enabled")
    print("----------------------------------------------")
    print(f"Particles                  : {N}")
    print(f"SPH smoothing radius h      : {h}")
    print(f"Uniform grid cell size      : {CELL_SIZE}")
    print(f"Uniform grid resolution     : {GRID_NX} x {GRID_NY} x {GRID_NZ}")
    print(f"Max particles per cell      : {MAX_PARTICLES_PER_CELL}")
    print(f"Level set grid resolution   : {LEVELSET_RES} x {LEVELSET_RES} x {LEVELSET_RES}")
    print(f"Level set support cells     : {LEVELSET_SUPPORT_CELLS}")
    print(f"Max frames                  : {MAX_FRAMES}")
    print(f"Save interval               : {SAVE_INTERVAL}")
    print(f"Save directory              : {SAVE_DIR}")
    print("==============================================")

    # frame 0 저장하고 싶으면 여기서 저장
    save_levelset_npz(0)

    for frame in range(1, MAX_FRAMES + 1):
        simulate_one_frame()

        if frame % SAVE_INTERVAL == 0:
            save_levelset_npz(frame)

    print("==============================================")
    print("Simulation finished.")
    print(f"Saved level set files to: {SAVE_DIR}")
    print("==============================================")


if __name__ == "__main__":
    main()

In [ ]:
# ==========================================
# 1. 모듈 임포트 및 Taichi 엔진 초기화
# ==========================================
import taichi as ti
from modules.config import config
from modules.taichi_fluid_solver import TaichiFluidSolver
from modules.dataset import SDFDataset

ti.init(arch=ti.gpu)

# ==========================================
# 2. 유체 시뮬레이터 세팅
# ==========================================
res = config.resolution            
domain_size = config.domain_size

solver = TaichiFluidSolver(res=res, domain_size=domain_size)
solver.setup_initial_fluid_block()

print(f"✅ 준비된 활성 파티클 수: {solver.get_active_particle_count():,}개")

# ==========================================
# 3. (변경) 파티클 데이터셋 NPY 추출 실행!
# ==========================================
# 뷰어 대신 저장 함수를 호출합니다. (예: 200 프레임 추출)
SDFDataset.save_particles_to_npy(
    solver=solver, 
    output_dir="dataset4", 
    num_frames=200
)

In [ ]:
# ==========================================
# 1. 모듈 임포트 및 Taichi 엔진 초기화
# ==========================================
import os
import glob
import numpy as np
import taichi as ti

from modules.config import config
from modules.dataset import SDFDataset
# ==========================================
# 2. 경로 및 파라미터 세팅
# ==========================================
dataset_dir = "dataset4"
res = config.resolution
domain_size = config.domain_size
radius_ratio = 1.5  # 파티클 하나가 차지하는 SDF 두께 배수

# ==========================================
# 3. 파티클 파일 순회 및 SDF 변환 루프
# ==========================================
print(f"\n🚀 '{dataset_dir}' 폴더의 파티클 데이터를 SDF로 변환 시작...")

# 폴더 내의 모든 particles_*.npy 파일을 찾아서 이름순으로 정렬
particle_files = sorted(glob.glob(os.path.join(dataset_dir, "particles_*.npy")))

if not particle_files:
    print(f"⚠️ '{dataset_dir}' 폴더에 변환할 파티클 파일이 없습니다.")
else:
    for i, file_path in enumerate(particle_files):
        # A. 파티클 데이터 로드
        particles = np.load(file_path)
        
        # B. 초고속 SDF 변환 커널 실행
        sdf_grid = SDFDataset.convert_particles_to_sdf(
            particles_np=particles,
            res=res,
            domain_size=domain_size,
            radius_ratio=radius_ratio 
        )
        
        # C. 저장할 SDF 파일명 생성 (기존 프레임 번호 매칭)
        # 예: "particles_0010.npy" -> "0010" 추출 -> "sdf_grid_0010.npy" 생성
        filename = os.path.basename(file_path) 
        frame_str = filename.split('_')[1].split('.')[0] 
        sdf_filename = os.path.join(dataset_dir, f"sdf_grid_{frame_str}.npy")
        
        # D. 디스크에 저장
        np.save(sdf_filename, sdf_grid)
        
        # 진행 상황 출력 (10프레임 단위)
        if i % 10 == 0 or i == len(particle_files) - 1:
            print(f"   [{i+1}/{len(particle_files)}] {sdf_filename} 저장 완료 (파티클 수: {len(particles):,})")

    print("\n🎉 모든 파티클 데이터의 SDF 변환 및 저장이 완료되었습니다!")

In [ ]:
from modules.visualizer import save_mesh_as_obj, draw_sdf_mesh, view_obj_interactive

# 1. 확인할 프레임 번호 및 기본 경로 설정
frame_number = 100  # 확인하고 싶은 프레임 번호 입력
dataset_dir = "dataset4"

# 앞서 데이터셋 생성 시 사용한 자릿수(03d)에 맞춰 파일명 지정
npy_path = os.path.join(dataset_dir, f"sdf_grid_{frame_number:04d}.npy")
obj_path = os.path.join(dataset_dir, f"frame_{frame_number:04d}.obj")

# 2. SDF 데이터 로드 및 처리
if not os.path.exists(npy_path):
    print(f"❌ 경고: 파일을 찾을 수 없습니다 ({npy_path})")
else:
    print(f"[{frame_number:04d} 프레임] SDF 데이터를 로드합니다...")
    sdf_grid = np.load(npy_path)

    # 3. visualizer.py의 함수를 호출하여 OBJ 파일로 추출 및 저장
    success = save_mesh_as_obj(sdf_grid, filename=obj_path, level=0.0)


In [ ]:
import open3d as o3d
import os

frame_number = 100  # 확인하고 싶은 프레임 번호 입력
obj_path = os.path.join(dataset_dir, f"frame_{frame_number:04d}.obj")

# 4. OBJ 파일이 성공적으로 저장되었다면 뷰어 실행
view_obj_interactive(filename=obj_path)

## 3. sdf 계산 -CNN 학습

### 3.1 전처리 단계: 그리드 특징값(m_c) 계산

### 3.2 m_c 값 시각화

SDF Input : 학습이나 추론용 

In [ ]:
import torch
import numpy as np
import os
# 1. 모듈 임포트
from modules.config import config
from modules.sdf_generator import SDFGenerator
from modules.particle_sampler import sample_particles_poisson
from modules.sdf_network import FeatureConstruction
from modules.visualizer import visualize_simulation, visualize_particles_and_features

# ==========================================
# Setup: Config & 디바이스 초기화
# ==========================================
print(f"⚙️ Config 설정: Domain Size={config.domain_size}, Resolution={config.resolution}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==========================================
# Phase 1: SDF 도형 생성 및 래스터라이징
# ==========================================
print("\n[Phase 1] 🎲 랜덤 SDF 도형 생성 중...")
# 새로 만든 OOP 클래스를 사용합니다.
generator = SDFGenerator(config)

# random_sdf = generator.create_sample_shape() 
test_shape = create_multi_thickness_plates()
sdf_grid = generator.to_grid(test_shape)

print(f"✅ SDF Grid 생성 완료: Shape={sdf_grid.shape}")

# ==========================================
# Phase 2: Poisson Disk 샘플링
# ==========================================
print("\n[Phase 2] 🎯 파티클 샘플링 중...")
# config 객체를 두 번째 인자로 넘겨받도록 변경된 부분을 반영합니다.
particles = sample_particles_poisson(sdf_grid, config)

print(f"✅ 파티클 샘플링 완료: {len(particles)}개 생성됨 (목표: {config.num_particles}개)")

# ==========================================
# Phase 3: 특징 인코딩 (Feature Construction)
# ==========================================
print("\n[Phase 3] 🧠 모델 입력용 특징(m_c) 추출 중...")
# 함수형 형태를 유지한 원본 FeatureConstruction을 사용하되, dx는 config 참조
feature_constructor = FeatureConstruction(dx=config.dx, device=device)

particles_tensor = torch.tensor(particles, dtype=torch.float32, device=device)

# 파티클 위치 기반으로 동적 그리드 변환 
grid_nodes, m_c, grid_shape = feature_constructor(particles_tensor)
m_c_grid = m_c.reshape(grid_shape).cpu().numpy()

print(f"✅ 기하학적 특징 추출 완료")
print(f"   - Grid Shape: {grid_shape}")
print(f"   - m_c 통계: Min={m_c.min():.4f}, Max={m_c.max():.4f}")

# ==========================================
# Phase 4: 데이터 저장 (.npy)
# ==========================================
print("\n[Phase 4] 💾 생성된 데이터를 .npy 형식으로 저장 중...")

# 출력 폴더 생성 (선택 사항, 깔끔한 관리를 위해)
save_dir = "./"
os.makedirs(save_dir, exist_ok=True)

# 텐서로 남아있는 grid_nodes를 numpy로 변환
grid_nodes_np = grid_nodes.cpu().numpy() if isinstance(grid_nodes, torch.Tensor) else grid_nodes

# 각각 .npy 파일로 저장
np.save(os.path.join(save_dir, "sdf_grid_128.npy"), sdf_grid)
np.save(os.path.join(save_dir, "particles_128.npy"), particles)
np.save(os.path.join(save_dir, "mc_grid_128.npy"), m_c_grid)
# np.save(os.path.join(save_dir, "grid_nodes_64.npy"), grid_nodes_np)

print(f"✅ 모든 데이터 저장 완료! (저장 위치: ./{save_dir}/)")

Particle Input

In [ ]:

particles_tensor = torch.tensor(particles, dtype=torch.float32, device=device)

# 파티클 위치 기반으로 동적 그리드 변환 
grid_nodes, m_c, grid_shape = feature_constructor(particles_tensor)
m_c_grid = m_c.reshape(grid_shape).cpu().numpy()

print(f"✅ 기하학적 특징 추출 완료")
print(f"   - Grid Shape: {grid_shape}")
print(f"   - m_c 통계: Min={m_c.min():.4f}, Max={m_c.max():.4f}")

# ==========================================
# Phase 4: 데이터 저장 (.npy)
# ==========================================
print("\n[Phase 4] 💾 생성된 데이터를 .npy 형식으로 저장 중...")

# 출력 폴더 생성 (선택 사항, 깔끔한 관리를 위해)
save_dir = "./"
os.makedirs(save_dir, exist_ok=True)

# 텐서로 남아있는 grid_nodes를 numpy로 변환
grid_nodes_np = grid_nodes.cpu().numpy() if isinstance(grid_nodes, torch.Tensor) else grid_nodes


# 4-2. 파티클 분포와 네트워크의 m_c 추출 결과 뷰어
visualize_particles_and_features(
    particles=particles_tensor.cpu(),
    grid_nodes=grid_nodes.cpu(),
    m_c_grid=m_c_grid,
    title="Network Feature Distribution (m_c)"
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from modules.config import config
from modules.particle_sampler import sample_particles_poisson
from modules.visualizer import visualize_simulation

print("=== 현재 디렉토리의 sdf_grid_64.npy 시각화 및 파티클 샘플링 ===")

# 1. 원본 SDF 데이터 불러오기
file_path = "sdf_grid_64.npy"
try:
    sdf_grid = np.load(file_path)
    print(f"✅ 데이터 로드 성공: {file_path} (Shape: {sdf_grid.shape})")
except FileNotFoundError:
    print(f"❌ 파일을 찾을 수 없습니다: {file_path}")
    sdf_grid = None
    
if sdf_grid is not None:
    # 2. 파티클 샘플링
    print("\n[파티클 샘플링 중...]")
    particles = sample_particles_poisson(sdf_grid, config)
    print(f"✅ 추출된 파티클 갯수: {len(particles)}")
    # 3. 파티클 위치 기반으로 동적 그리드 변환 및 m_c 계산
    print("\n[DirectML 가속으로 m_c 특징값 계산 중...]")
    feature_constructor = FeatureConstruction(dx=config.dx, device=device)
    particles_tensor = torch.tensor(particles, dtype=torch.float32, device=device)
    
    # 🌟 기존에 구현되어 있는 통합 함수 호출!
    grid_nodes, m_c, grid_shape = feature_constructor(particles_tensor)
    
    # 3D 텐서(64x64x64)로 복원 후 numpy 배열로 변환
    mc_grid = m_c.reshape(grid_shape).cpu().numpy()
    
    # 4. 결과 저장 (mc_grid_64.npy)
    save_path = "mc_grid_64.npy"
    # np.save(save_path, mc_grid)
    print(f"✅ m_c 그리드 저장 완료: {save_path} (Shape: {mc_grid.shape})")
    # 5. m_c 3D 특징 맵 시각화
    print("\n[m_c 특징 공간 시각화]")
    fig = visualize_feature_grid(
        m_c_grid=m_c,             # flatten된 1D 텐서 그대로 투입
        grid_nodes=grid_nodes,    # 짝이 완벽하게 맞는 3D 좌표 노드
        title="Computed m_c Feature Grid (from sdf_grid_64.npy)"
    )
    plt.show()

In [ ]:
# 1. 확인할 데이터 인덱스 설정 (0 ~ 4)
sdf_file = f"sdf_grid_128.npy"
mc_file = f"mc_grid_128.npy"

# 2. 데이터 로드 완료
sdf_grid = np.load(sdf_file)
mc_grid = np.load(mc_file)
print(f"📥 m_c 데이터 로드: {mc_file} (Shape: {mc_grid.shape})")

fig2 = visualize_feature_grid(
    m_c_grid=mc_grid,         # 앞서 파이프라인에서 나온 mc_grid 텐서 그대로 삽입
    grid_nodes=grid_nodes,    # 앞서 파이프라인에서 나온 grid_nodes 텐서 그대로 삽입
    title=f"Input m_c Feature Grid"
)

### 3.3 3D CNN 네트워크를 통한 SDF 값 추론

### 3.4 학습

디바이스/데이터로더 설정 셀

훈련 반복문 셀

In [ ]:
import time
import numpy as np
import torch
import torch.optim as optim
from tqdm import tqdm

from modules.dataset import SDFDataset, DataLoader
from modules.sdf_network import SDFNetwork, PolynomialRegularizedSDFLoss, SDFReconstruction, FeatureConstruction
from modules.config import config
from modules.visualizer import visualize_simulation

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"⚡ 작동 디바이스: {device}")

# ==========================================
# 📦 1. 데이터 로드 (Narrow Band 적용)
# ==========================================
print("\n[데이터 로드 시작]")

# 🚨 [추가] 데이터셋 내부에서 파티클을 m_c로 변환할 수 있게 생성자 전달
feature_constructor = FeatureConstruction(dx=config.dx, device=device)

train_dataset = SDFDataset(
    data_dir="dataset5",   # 🚨 경로 확인 (dataset4)
    patch_size=8,
    in_memory=True, 
    use_narrow_band=True, 
    use_bg_sample=False,     
    bg_sample_ratio=0.10,
    feature_constructor=feature_constructor  # 🚨 [핵심] 여기서 넘겨줍니다!
)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=0)

In [ ]:
import time
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# 사용자 정의 모듈 임포트 (경로는 본인 환경에 맞게 유지)
from modules.sdf_network import SDFNetwork, SDFReconstruction
from modules.config import config
from modules.visualizer import visualize_simulation

print("=== [Step 5] 저장된 가중치 불러오기 및 다이렉트 SDF 추론 ===")

# -------------------------------------------------------------------
# [핵심 수정 사항] 진짜 신경망 모델(SDFNetwork)을 불러옵니다!
# -------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. 빈 모델 껍데기(구조) 생성
model = SDFNetwork().to(device)

# 2. 가중치 파일 경로 및 로드
weights_path = "sdf_network_cpu.pth"
model.load_state_dict(torch.load(weights_path, map_location=device))

# 3. 평가(eval) 모드로 전환
model.eval()

print(f"✅ '{weights_path}' 가중치 로드 완료! (Device: {device})")


In [ ]:
# 공통 학습 파라미터 세팅 (빠른 실험을 위해 데이터의 50%만 학습)
num_epochs = 5
max_batches = len(train_loader)

In [ ]:

# ==========================================
# 🧪 [실험 1] 다항식 정규화(Polynomial) + MSE 
# ==========================================
print("\n" + "="*55)
print(">> 🧪 [실험 1] 다항식 정규화(Poly) + MSE 모드 학습 시작")
print("="*55)

model_poly = SDFNetwork().to(device)
optimizer_poly = optim.Adam(model_poly.parameters(), lr=0.0002)
criterion_poly = PolynomialRegularizedSDFLoss(lambda_reg=1.0, use_poly_loss=True).to(device)

start_time = time.time()
model_poly.train_step(
    train_loader=train_loader,
    optimizer=optimizer_poly,
    criterion=criterion_poly,
    device=device,
    num_epochs=num_epochs,
    max_batches_per_epoch=max_batches
)
print(f"✅ 실험 1 학습 완료! (소요 시간: {time.time() - start_time:.1f}초)")




In [ ]:
# ==========================================
# 🧪 [실험 2] 순수 MSE 모드 (정규화 제거 비교용)
# ==========================================
print("\n" + "="*55)
print(">> 🧪 [실험 2] 순수 MSE 모드 학습 시작 (Poly OFF)")
print("="*55)

# 🚨 이전 모델의 기억을 지우고 완전히 새로운 모델 생성
model_mse = SDFNetwork().to(device)
optimizer_mse = optim.Adam(model_mse.parameters(), lr=0.0002)
# 🚨 use_poly_loss=False (정규화 OFF)
criterion_mse = PolynomialRegularizedSDFLoss(use_poly_loss=False).to(device)

start_time = time.time()
model_mse.train_step(
    train_loader=train_loader,
    optimizer=optimizer_mse,
    criterion=criterion_mse,
    device=device,
    num_epochs=num_epochs,
    max_batches_per_epoch=max_batches
)
print(f"✅ 실험 2 학습 완료! (소요 시간: {time.time() - start_time:.1f}초)")



In [ ]:
# 1. 평가용 데이터 로드 
# 🚨 핵심 변경: mc_grid.npy 대신 원본 파티클 좌표를 직접 불러옵니다!
particles_np = np.load("particles_128.npy")
particles_tensor = torch.tensor(particles_np, dtype=torch.float32, device=device)

sdf_grid_gt = np.load("sdf_grid_128.npy")  # 정답지 (기본 해상도, 비교용)
# 추론 및 시각화를 위한 평가용 데이터 로드
m_c_grid_np = np.load("mc_grid_128.npy")
m_c_grid_tensor = torch.tensor(m_c_grid_np, dtype=torch.float32)

print(">> 실험 1 추론")
reconstructor_poly = SDFReconstruction(dx=config.dx, device=device)
reconstructor_poly.network = model_poly  # 학습된 모델 장착
pred_sdf_poly = reconstructor_poly.inference(m_c_grid_tensor, patch_size=8, batch_size=2048)

print(">> 실험 2 추론 중...")
reconstructor_mse = SDFReconstruction(dx=config.dx, device=device)
reconstructor_mse.network = model_mse  # 두 번째로 학습된 모델 장착
pred_sdf_mse = reconstructor_mse.inference(m_c_grid_tensor, patch_size=8, batch_size=2048)


In [ ]:
# ==========================================
# 💾 모델 가중치 저장하기
# ==========================================
weights_path_mse = "sdf_network_mse.pth"

# 모델의 상태(가중치)를 딕셔너리 형태로 저장합니다.
torch.save(reconstructor_poly.network.state_dict(), "sdf_network_poly_27ch.pth")
print(f"💾 MSE 모델 가중치가 안전하게 저장되었습니다: {weights_path_mse}")

In [ ]:

# ==========================================
# 📊 최종 결과 시각화 및 비교
# ==========================================
print("\n=== [Step 6] 결과물 시각화 (Ablation 비교) ===")

# 1. 정답지 (도넛 모양)
visualize_simulation(sdf_grid=sdf_grid_gt, domain_size=config.domain_size, title="1. Ground Truth (Real SDF)")

# 2. 논문 세팅 결과 (매끄러운 곡면)
visualize_simulation(sdf_grid=pred_sdf_poly.cpu().numpy(), domain_size=config.domain_size, title="2. Polynomial + MSE (Smooth)")

# 3. 정규화 끈 결과 (쭈글쭈글한 표면)
visualize_simulation(sdf_grid=pred_sdf_mse.cpu().numpy(), domain_size=config.domain_size, title="3. Pure MSE Only (Jagged)")

In [ ]:
print("\n" + "="*55)
print(">> 🎯 [실험 1] Poly 모델 고해상도(Staggered) 추론 시작")
print("="*55)
reconstructor_poly = SDFReconstruction(dx=config.dx, device=device)
reconstructor_poly.network = model_poly  # 첫 번째로 학습된 모델 장착

start_time = time.time()
# 🚨 inference 대신 staggered_inference 호출!
# 내부적으로 Pruning이 8번 작동하여 연산 속도를 크게 단축시킵니다.
pred_sdf_poly_high_res = reconstructor_poly.staggered_inference(
    particles_tensor, 
    patch_size=8, 
    batch_size=2048
)
print(f"   -> ⏱️ Poly 모델 총 소요 시간: {time.time() - start_time:.1f}초\n")


print("="*55)
print(">> 🎯 [실험 2] MSE 모델 고해상도(Staggered) 추론 시작")
print("="*55)
reconstructor_mse = SDFReconstruction(dx=config.dx, device=device)
reconstructor_mse.network = model_mse  # 두 번째로 학습된 모델 장착

start_time = time.time()
pred_sdf_mse_high_res = reconstructor_mse.staggered_inference(
    particles_tensor, 
    patch_size=8, 
    batch_size=2048
)
print(f"   -> ⏱️ MSE 모델 총 소요 시간: {time.time() - start_time:.1f}초\n")

In [ ]:
# ==========================================
# 📊 결과 시각화 및 비교
# ==========================================
print("=== 📊 고해상도 결과물 시각화 ===")
print("💡 참고: Ground Truth는 기본 해상도(N)이며, 예측 결과는 2배 해상도(2N)입니다.")

# 1. 정답지 (기본 해상도)
visualize_simulation(
    sdf_grid=sdf_grid_gt, 
    domain_size=config.domain_size, 
    title="[Ground Truth] Base Resolution"
)

# 2. 논문 세팅 결과 (2배 해상도 + 정규화로 매우 매끄러움)
visualize_simulation(
    sdf_grid=pred_sdf_poly_high_res.cpu().numpy(), 
    domain_size=config.domain_size, 
    title="[Poly + MSE] High Res 2x (Staggered)"
)

# 3. 정규화 끈 결과 (2배 해상도지만 표면이 울퉁불퉁할 수 있음)
visualize_simulation(
    sdf_grid=pred_sdf_mse_high_res.cpu().numpy(), 
    domain_size=config.domain_size, 
    title="[Pure MSE] High Res 2x (Staggered)"
)

In [ ]:
# ==========================================
# 🎯 [추가 실험] 특정 데이터 추론 및 비교
# ==========================================
# 🚨 여기서 원하는 데이터의 인덱스 번호만 바꾸면 끝입니다! (예: 1, 9, 10)
target_idx = 10

# 0을 채워서 자동으로 3자리 문자열로 만듭니다. (예: 1 -> "001", 15 -> "015")
shape_name = f"{target_idx:03d}" 

print("\n" + "="*55)
print(f">> 🎯 특정 테스트 데이터(Shape {shape_name}) 추론 시작")
print("="*55)

# 1. 지정된 타겟 & 특징 그리드 로드 (전처리 완전 생략!)
# 파일 경로가 자동으로 세팅됩니다.
sdf_path = f"dataset4/sdf_grid_{shape_name}.npy"
mc_path = f"dataset4/mc_grid_{shape_name}.npy"

target_sdf = np.load(sdf_path)
m_c_grid_np = np.load(mc_path)

# 텐서 변환
m_c_grid_tensor = torch.tensor(m_c_grid_np, dtype=torch.float32)

# 2. [Poly 모드] 이미 학습된 Poly 모델로 초고속 추론
print(f">> Poly 모델(정규화 ON)로 Shape {shape_name} 추론 중...")
pred_sdf_poly = reconstructor_poly.inference(m_c_grid_tensor, patch_size=8, batch_size=2048)

# 3. [MSE 모드] 이미 학습된 MSE 모델로 초고속 추론
print(f">> MSE 모델(정규화 OFF)로 Shape {shape_name} 추론 중...")
pred_sdf_mse = reconstructor_mse.inference(m_c_grid_tensor, patch_size=8, batch_size=2048)

# ==========================================
# 📊 결과 시각화 및 비교
# ==========================================
print(f"\n=== [Step 7] Shape {shape_name} 결과물 시각화 (Unseen Data Test) ===")

# 1. 정답지 (진짜 모습)
visualize_simulation(
    sdf_grid=target_sdf, 
    domain_size=config.domain_size, 
    title=f"[Shape {shape_name}] Ground Truth"
)

# 2. 논문 세팅 결과 (정규화가 들어간 매끄러운 예측 결과)
visualize_simulation(
    sdf_grid=pred_sdf_poly.cpu().numpy(), 
    domain_size=config.domain_size, 
    title=f"[Shape {shape_name}] Polynomial + MSE (Smooth)"
)

# 3. 정규화 끈 결과 (정규화가 없을 때의 쭈글쭈글한 예측 결과)
visualize_simulation(
    sdf_grid=pred_sdf_mse.cpu().numpy(), 
    domain_size=config.domain_size, 
    title=f"[Shape {shape_name}] Pure MSE Only (Jagged)"
)

In [ ]:
# ==========================================
# 🎯 [추가 실험] 특정 데이터 고해상도(Staggered) 추론 및 비교
# ==========================================
# 🚨 여기서 원하는 데이터의 인덱스 번호만 바꾸면 끝입니다! (예: 1, 9, 10)
target_idx = 30

# 0을 채워서 자동으로 3자리 문자열로 만듭니다. (예: 1 -> "001", 15 -> "015")
shape_name = f"{target_idx:03d}" 

print("\n" + "="*55)
print(f">> 🎯 특정 테스트 데이터(Shape {shape_name}) 고해상도(2배) 추론 시작")
print("="*55)

# 1. 원본 파티클 데이터 & 정답 타겟 그리드 로드
# 🚨 핵심 변경: 이전처럼 mc_grid를 쓰지 않고, 최근 생성한 dataset4의 particles를 불러옵니다!
sdf_path = f"dataset4/sdf_grid_{shape_name}.npy"
particles_path = f"dataset4/particles_{shape_name}.npy"

target_sdf = np.load(sdf_path)          # (N, N, N) 기본 해상도 정답지
particles_np = np.load(particles_path)  # 원본 파티클 좌표

# 파티클을 텐서로 변환하여 GPU로 전송
particles_tensor = torch.tensor(particles_np, dtype=torch.float32, device=device)

# 2. [Poly 모드] 엇갈린 그리드(Staggered) 다중 추론 
print(f">> Poly 모델(정규화 ON)로 Shape {shape_name} 고해상도 뻥튀기 중...")
start_time = time.time()
pred_sdf_poly_high_res = reconstructor_poly.staggered_inference(
    particles_tensor, patch_size=8, batch_size=2048
)
print(f"   -> ⏱️ 소요 시간: {time.time() - start_time:.1f}초")

# 3. [MSE 모드] 엇갈린 그리드(Staggered) 다중 추론
print(f">> MSE 모델(정규화 OFF)로 Shape {shape_name} 고해상도 뻥튀기 중...")
start_time = time.time()
pred_sdf_mse_high_res = reconstructor_mse.staggered_inference(
    particles_tensor, patch_size=8, batch_size=2048
)
print(f"   -> ⏱️ 소요 시간: {time.time() - start_time:.1f}초")

# ==========================================
# 📊 결과 시각화 및 비교
# ==========================================
print(f"\n=== [Step 7] Shape {shape_name} 고해상도 결과물 시각화 ===")
print("💡 참고: Ground Truth는 기본 해상도(N)이며, 예측 결과는 2배 해상도(2N)입니다.")

# 1. 정답지 (기본 해상도)
visualize_simulation(
    sdf_grid=target_sdf, 
    domain_size=config.domain_size, 
    title=f"[Shape {shape_name}] Ground Truth (Base Res)"
)

# 2. 논문 세팅 결과 (2배 해상도 + 정규화로 매우 매끄러움)
visualize_simulation(
    sdf_grid=pred_sdf_poly_high_res.cpu().numpy(), 
    domain_size=config.domain_size, 
    title=f"[Shape {shape_name}] Poly + MSE (High Res 2x)"
)

# 3. 정규화 끈 결과 (2배 해상도지만 표면이 울퉁불퉁할 수 있음)
visualize_simulation(
    sdf_grid=pred_sdf_mse_high_res.cpu().numpy(), 
    domain_size=config.domain_size, 
    title=f"[Shape {shape_name}] Pure MSE (High Res 2x)"
)

In [ ]:
def load_particles_from_txt(txt_path, device='cpu'):
    """
    x y z 형태로 저장된 파티클 .txt 파일을 읽어 PyTorch 텐서로 반환합니다.
    (축 방향 수정 및 -1~1 스케일 정규화 포함)
    """
    import os
    import numpy as np
    import torch

    if not os.path.exists(txt_path):
        raise FileNotFoundError(f"❌ 파일을 찾을 수 없습니다: {txt_path}")
        
    print(f"📄 파티클 파일 로드 중: {txt_path}")
    
    valid_particles = []
    error_count = 0
    
    with open(txt_path, 'r') as f:
        for line_idx, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
                
            parts = line.split()
            if len(parts) == 3:
                try:
                    # 1. 텍스트에서 숫자 추출
                    x = float(parts[0])
                    y = float(parts[1])
                    z = float(parts[2])
                    
                    # 🚨 [해결책 1] Y축과 Z축 스왑 (Coordinate System 교정)
                    # 기존 [x, y, z] 대신 [x, z, y] 순서로 리스트에 담습니다.
                    valid_particles.append([x, z, y])
                    
                except ValueError:
                    error_count += 1
            else:
                if error_count == 0: 
                    print(f"  ⚠️ [형식 무시됨] Line {line_idx+1}: '{line}'")
                error_count += 1

    if not valid_particles:
        raise ValueError("❌ 유효한 파티클(x y z) 데이터를 하나도 찾지 못했습니다.")
        
    particles_np = np.array(valid_particles, dtype=np.float32)
    
    # =======================================================
    # 🚨 [해결책 2] 0~1 공간을 -1~1 공간으로 뻥튀기 (Scale Remapping)
    # 공식: New_Value = (Old_Value * 2.0) - 1.0
    # 이렇게 하면 0.0은 -1.0이 되고, 0.5는 0.0(중앙)이 되며, 1.0은 1.0이 됩니다.
    # =======================================================
    print(f"🔍 변환 전 파티클 범위: Min {particles_np.min():.3f} ~ Max {particles_np.max():.3f}")
    
    particles_np = (particles_np) - 0.5
    
    print(f"🚀 변환 후 파티클 범위: Min {particles_np.min():.3f} ~ Max {particles_np.max():.3f} (그리드 최적화 완료!)")
    # =======================================================

    print(f"✅ 로드 완료! 정상 파티클: {particles_np.shape[0]:,}개 (건너뛴 줄: {error_count}개)")
    
    # GPU 텐서로 변환하여 반환
    particles_tensor = torch.tensor(particles_np, dtype=torch.float32, device=device)
    return particles_tensor

In [ ]:
# 🚨 읽어올 파티클 텍스트 파일 경로
txt_particle_path = "frame_094.txt"
particles_tensor = load_particles_from_txt(txt_particle_path, device=device)

# 파티클 위치 기반으로 동적 그리드 변환 
grid_nodes, m_c, grid_shape = feature_constructor(particles_tensor)
m_c_grid = m_c.reshape(grid_shape).cpu().numpy()

print(f"✅ 기하학적 특징 추출 완료")
print(f"   - Grid Shape: {grid_shape}")
print(f"   - m_c 통계: Min={m_c.min():.4f}, Max={m_c.max():.4f}")

# ==========================================
# Phase 4: 데이터 저장 (.npy)
# ==========================================
# print("\n[Phase 4] 💾 생성된 데이터를 .npy 형식으로 저장 중...")

# # 출력 폴더 생성 (선택 사항, 깔끔한 관리를 위해)
save_dir = "./"
os.makedirs(save_dir, exist_ok=True)

# # 텐서로 남아있는 grid_nodes를 numpy로 변환
grid_nodes_np = grid_nodes.cpu().numpy() if isinstance(grid_nodes, torch.Tensor) else grid_nodes


# 4-2. 파티클 분포와 네트워크의 m_c 추출 결과 뷰어
visualize_particles_and_features(
    particles=particles_tensor.cpu(),
    grid_nodes=grid_nodes.cpu(),
    m_c_grid=m_c_grid,
    title="Network Feature Distribution (m_c)"
)


In [ ]:
# 만약 정답지(GT) SDF가 있다면 경로를 적고, 없다면 None으로 두세요.
gt_sdf_path = None # 예: "dataset5/sdf_grid_001.npy"
txt_particle_path = "frame_094.txt"
print("\n" + "="*55)
print(f">> 🎯 외부 파티클 데이터({txt_particle_path}) 고해상도(2배) 추론 시작")
print("="*55)

# 1. 텍스트 파티클 데이터 텐서로 로드
particles_tensor = load_particles_from_txt(txt_particle_path, device=device)

# 정답지 로드 (있는 경우에만)
target_sdf = None
if gt_sdf_path and os.path.exists(gt_sdf_path):
    target_sdf = np.load(gt_sdf_path)
    print("📊 정답지(Ground Truth) SDF를 로드했습니다.")
else:
    print("💡 정답지(Ground Truth) 없이 모델 예측만 수행합니다.")

# 2. [Poly 모드] 엇갈린 그리드(Staggered) 다중 추론 
print("\n>> Poly 모델(정규화 ON)로 고해상도 뻥튀기 중...")
start_time = time.time()
pred_sdf_poly_high_res = reconstructor_poly.staggered_inference(
    particles_tensor, patch_size=8, batch_size=2048
)
print(f"   -> ⏱️ 소요 시간: {time.time() - start_time:.1f}초")

# 3. [MSE 모드] 엇갈린 그리드(Staggered) 다중 추론
# print("\n>> MSE 모델(정규화 OFF)로 고해상도 뻥튀기 중...")
# start_time = time.time()
# pred_sdf_mse_high_res = reconstructor_mse.staggered_inference(
#     particles_tensor, patch_size=8, batch_size=2048
# )
# print(f"   -> ⏱️ 소요 시간: {time.time() - start_time:.1f}초")

# ==========================================
# 📊 결과 시각화 및 비교
# ==========================================
print(f"\n=== [Step 3] 예측 결과물 시각화 ===")

# 1. 정답지 (존재할 때만 렌더링)
if target_sdf is not None:
    visualize_simulation(
        sdf_grid=target_sdf, 
        domain_size=config.domain_size, 
        title="[Ground Truth] Base Res"
    )

# 2. 논문 세팅 결과 (2배 해상도 + 정규화로 매우 매끄러움)
visualize_simulation(
    sdf_grid=pred_sdf_poly_high_res.cpu().numpy(), 
    domain_size=config.domain_size, 
    title="[Poly + MSE] High Res 2x (txt)"
)

# 3. 정규화 끈 결과 (2배 해상도지만 표면이 울퉁불퉁할 수 있음)
# visualize_simulation(
#     sdf_grid=pred_sdf_mse_high_res.cpu().numpy(), 
#     domain_size=config.domain_size, 
#     title="[Pure MSE] High Res 2x (txt)"
# )

## 4. SDF 값 시각화 (최종 결과)

4.2 obj로 저장(mc 알고리즘)

In [ ]:
from modules.visualizer import save_mesh_as_obj # 시각화용
save_mesh_as_obj(target_sdf, filename="gt.obj")
save_mesh_as_obj(pred_sdf_poly.cpu().numpy(), filename="train_shape.obj")
save_mesh_as_obj(pred_sdf_mse.cpu().numpy(), filename="test_shape.obj")


In [ ]:
from modules.visualizer import save_mesh_as_obj # 시각화용
# save_mesh_as_obj(target_sdf, filename="gt_high.obj")
save_mesh_as_obj(pred_sdf_poly_high_res.cpu().numpy(), filename="pred_sdf_poly_high_res.obj")
# save_mesh_as_obj(pred_sdf_mse_high_res.cpu().numpy(), filename="pred_sdf_mse_high_res.obj")

npy 파일 이용한 sdf 시각화

In [ ]:
from modules.visualizer import visualize_npy
# 2. 데이터 시각화 (Phase 2)
print("\n--- Phase 2: 시각화 및 저장 ---")

# SDF 파일 확인
visualize_npy("sdf_grid_64.npy")

# 파티클 파일 확인
visualize_npy("particles.npy")


변수(객체) 이용한 시각화(런타임에 있는 메모리)

In [ ]:
from modules.visualizer import visualize_simulation

# SDF만 보기 (기존 plot_slice + 3D)
visualize_simulation(sdf_grid=sdf_grid)

# 파티클까지 겹쳐서 보기 (기존 plot_particles_and_sdf_slice + 3D)
visualize_simulation(sdf_grid=sdf_grid, particles=particles)

obj 인터랙티브 뷰어

In [ ]:
from modules.visualizer import view_obj_interactive
# OBJ 파일 인터랙티브 뷰어
view_obj_interactive("train_shape.obj")

In [ ]:
from modules.visualizer import view_obj_interactive
# OBJ 파일 인터랙티브 뷰어
view_obj_interactive("test_shape.obj")

In [ ]:
from modules.visualizer import view_obj_interactive
    # OBJ 파일 인터랙티브 뷰어
view_obj_interactive("pred_sdf_poly_high_res.obj")

In [ ]:
import numpy as np
from modules.visualizer import view_obj_and_particles_interactive

# 1. 파일에서 넘파이 배열 읽어오기
# particles_np = np.load("frame_001.npy")

particles_np = load_particles_from_txt("frame_094.txt")
# 🚨 변경된 부분: x, y, z에 각각 0.5 오프셋 추가
particles_np += 0.5

# 2. 뷰어에 메쉬와 파티클 배열 함께 전달하기
view_obj_and_particles_interactive("pred_sdf_poly_high_res.obj", particles_np, 128, 1.0)